In [0]:
create streaming table orders_bronze 
as
select *, _metadata.file_path as file_name,current_timestamp() as load_time 
from cloud_files('/Volumes/workspace/landing/landing/orders/','csv',map('cloudFiles.inferColumnTypes','True'));

In [0]:
create streaming table orders_silver_cleaned
( constraint valid_orders expect(id is not null)
  on violation drop row
) as
select orderid as id,orderdate as order_date,totalamount as total_amount,status,file_name,load_time
from stream(orders_bronze) 

In [0]:
create streaming table orders_silver;
create flow orders_silver_flow as auto cdc into orders_silver
from stream(orders_silver_cleaned)
keys(id)
sequence by load_time;

In [0]:
create materialized view city_wise_sales_gold
as 
select city,sum(total_amount) as total_sales
from live.orders_silver o
join live.customers_silver c
on o.id=c.customer_id
group by city